# LESSON 6.7: Constrained Least Squares Filtering and Final Comparison
## Image Restoration

In this lesson:
- Constrained Least Squares (CLS) filtering: theory and derivation
- The Laplacian regularization approach
- Choosing the regularization parameter γ
- Geometric mean filter
- Comprehensive comparison: Inverse vs Wiener vs CLS
- Biomedical imaging application: CT/MRI restoration
- Summary of all restoration methods

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Motivation: Limitations of Wiener Filter

The Wiener filter requires knowledge of:
- The degradation function $H(u,v)$
- The **noise power spectrum** $S_\eta(u,v)$
- The **signal power spectrum** $S_f(u,v)$ (which requires the original image!)

In practice, $S_f$ is unknown (we're trying to restore it!). The parametric Wiener filter approximates $S_\eta / S_f$ with a constant $K$, but this is a rough approximation.

### The CLS Alternative:
The **Constrained Least Squares (CLS)** filter requires only:
- The degradation function $H(u,v)$
- The **noise mean** and **noise variance** $\sigma_\eta^2$

These are much easier to estimate from the image itself!

---
## 2. Constrained Least Squares Filter Theory

### Objective:
Find $\hat{f}$ that minimizes the **smoothness criterion** (Laplacian energy) subject to a **noise constraint**:

$$\min_{\hat{f}} \| \nabla^2 \hat{f} \|^2 \quad \text{subject to} \quad \| g - H\hat{f} \|^2 = \| \eta \|^2$$

Where:
- $\nabla^2 \hat{f}$ = Laplacian of the restored image (measures roughness)
- $\| g - H\hat{f} \|^2$ = residual (difference between observed and model)
- $\| \eta \|^2$ = expected noise power

### Interpretation:
Find the **smoothest possible** image $\hat{f}$ that is still **consistent** with the observed data and noise level.

### Solution in the Frequency Domain:

$$\boxed{\hat{F}(u,v) = \frac{H^*(u,v)}{|H(u,v)|^2 + \gamma |P(u,v)|^2} \cdot G(u,v)}$$

Where:
- $P(u,v)$ = DFT of the **Laplacian operator**: $p = \begin{bmatrix} 0 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 0 \end{bmatrix}$
- $\gamma$ = **regularization parameter** (similar role to $K$ in Wiener)

### Comparison with Wiener:

| | Wiener | CLS |
|---|---|---|
| Formula | $\frac{H^*}{|H|^2 + K}$ | $\frac{H^*}{|H|^2 + \gamma|P|^2}$ |
| Regularization | Constant $K$ | **Frequency-dependent** $\gamma|P(u,v)|^2$ |
| Required knowledge | $S_\eta/S_f$ or $K$ | $\sigma_\eta^2$ (noise variance) |
| Smoothness prior | None (implicit) | Explicit (Laplacian) |

In [ ]:
# Helper functions

def create_test_image(size=256):
    """Create a synthetic biomedical phantom image."""
    img = np.ones((size, size), dtype=np.float64) * 30
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    
    body = ((X - cx) / 100) ** 2 + ((Y - cy) / 80) ** 2 <= 1
    img[body] = 120
    organ1 = ((X - cx + 30) / 35) ** 2 + ((Y - cy + 10) / 45) ** 2 <= 1
    img[organ1] = 170
    organ2 = ((X - cx - 35) / 25) ** 2 + ((Y - cy - 15) / 30) ** 2 <= 1
    img[organ2] = 80
    for (sx, sy, sr) in [(cx-20, cy+30, 5), (cx+40, cy-25, 4), (cx-50, cy-20, 3)]:
        spot = (X - sx) ** 2 + (Y - sy) ** 2 <= sr ** 2
        img[spot] = 240
    for i in range(cy-40, cy+40):
        j = int(cx + 20 * np.sin(2 * np.pi * i / 50))
        if 0 <= j < size and 0 <= i < size:
            img[i, max(0,j-1):min(size,j+2)] = 200
    return img


def compute_psnr(original, restored):
    """Compute PSNR in dB."""
    mse = np.mean((original - restored) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(255.0 ** 2 / mse)


def atmospheric_turbulence_otf(P, Q, k=0.0025):
    """Atmospheric turbulence degradation OTF."""
    u = np.arange(P) - P/2
    v = np.arange(Q) - Q/2
    U, V = np.meshgrid(u, v, indexing='ij')
    H = np.exp(-k * (U**2 + V**2)**(5/6))
    return H


def motion_blur_otf(P, Q, a=0.1, b=0.0, T=1.0):
    """Uniform linear motion blur OTF."""
    u = np.arange(P) - P/2
    v = np.arange(Q) - Q/2
    U, V = np.meshgrid(u, v, indexing='ij')
    arg = np.pi * (U * a + V * b)
    arg_safe = np.where(np.abs(arg) < 1e-10, 1e-10, arg)
    H = (T / arg_safe) * np.sin(arg_safe) * np.exp(-1j * arg_safe)
    H[np.abs(arg) < 1e-10] = T
    return H


def wiener_filter(G, H, K):
    """Parametric Wiener filter."""
    H_conj = np.conj(H)
    W = H_conj / (np.abs(H)**2 + K)
    F_hat = W * G
    restored = np.real(np.fft.ifft2(np.fft.ifftshift(F_hat)))
    return np.clip(restored, 0, 255)


def inverse_filter(G, H, epsilon=1e-3):
    """Pseudoinverse filter."""
    H_safe = np.where(np.abs(H) > epsilon, H, epsilon * np.exp(1j * np.angle(H)))
    F_hat = G / H_safe
    restored = np.real(np.fft.ifft2(np.fft.ifftshift(F_hat)))
    return np.clip(restored, 0, 255)


original = create_test_image(256)
M, N = original.shape
print(f"Test image: {M}×{N}")

In [ ]:
# Implement the CLS filter

def cls_filter(G, H, gamma, P_laplacian=None):
    """
    Constrained Least Squares (CLS) filter.
    
    F_hat = (H* / (|H|^2 + gamma * |P|^2)) * G
    
    Parameters:
        G: DFT of degraded image (centered)
        H: degradation OTF (centered)
        gamma: regularization parameter
        P_laplacian: DFT of Laplacian operator (if None, computed internally)
    Returns:
        Restored image
    """
    P_h, Q_w = G.shape
    
    if P_laplacian is None:
        # Create Laplacian kernel and compute its DFT
        laplacian = np.array([[0, -1, 0],
                              [-1, 4, -1],
                              [0, -1, 0]], dtype=np.float64)
        # Pad to image size (center at (0,0))
        lap_padded = np.zeros((P_h, Q_w), dtype=np.float64)
        for i in range(3):
            for j in range(3):
                ni = (i - 1) % P_h
                nj = (j - 1) % Q_w
                lap_padded[ni, nj] = laplacian[i, j]
        P_laplacian = np.fft.fftshift(np.fft.fft2(lap_padded))
    
    H_conj = np.conj(H)
    H_mag2 = np.abs(H) ** 2
    P_mag2 = np.abs(P_laplacian) ** 2
    
    W = H_conj / (H_mag2 + gamma * P_mag2)
    F_hat = W * G
    
    restored = np.real(np.fft.ifft2(np.fft.ifftshift(F_hat)))
    return np.clip(restored, 0, 255)


# Precompute the Laplacian DFT for reuse
laplacian = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float64)
lap_padded = np.zeros((M, N), dtype=np.float64)
for i in range(3):
    for j in range(3):
        ni = (i - 1) % M
        nj = (j - 1) % N
        lap_padded[ni, nj] = laplacian[i, j]
P_lap = np.fft.fftshift(np.fft.fft2(lap_padded))

# Visualize the Laplacian in the frequency domain
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(laplacian, cmap='RdBu', vmin=-4, vmax=4)
axes[0].set_title('Laplacian Kernel p(x,y)', fontsize=12)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{laplacian[i,j]:.0f}', ha='center', va='center', fontsize=14)

axes[1].imshow(np.abs(P_lap), cmap='hot')
axes[1].set_title('|P(u,v)| — Laplacian in Frequency Domain\n(High-pass: penalizes high frequencies)', fontsize=12)
axes[1].axis('off')

plt.suptitle('The Laplacian Operator Used in CLS Regularization',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("|P(u,v)| is large at HIGH frequencies and zero at the center.")
print("This means CLS penalizes high-frequency content (noise) more than low-frequency (signal).")

In [ ]:
# Effect of gamma on CLS filter

# Degrade with atmospheric turbulence + noise
k_turb = 0.005
H_turb = atmospheric_turbulence_otf(M, N, k_turb)
F_orig = np.fft.fftshift(np.fft.fft2(original))

np.random.seed(42)
noise_sigma = 5
degraded = np.real(np.fft.ifft2(np.fft.ifftshift(H_turb * F_orig)))
degraded += np.random.normal(0, noise_sigma, degraded.shape)
degraded = np.clip(degraded, 0, 255)

G = np.fft.fftshift(np.fft.fft2(degraded))

# Apply CLS with different gamma values
gamma_values = [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1]

fig, axes = plt.subplots(2, 6, figsize=(22, 7))

for i, gamma in enumerate(gamma_values):
    restored = cls_filter(G, H_turb, gamma, P_lap)
    psnr = compute_psnr(original, restored)
    
    # Show the CLS filter transfer function
    W_cls = np.abs(np.conj(H_turb) / (np.abs(H_turb)**2 + gamma * np.abs(P_lap)**2))
    axes[0, i].imshow(W_cls, cmap='gray')
    axes[0, i].set_title(f'|W| γ={gamma}', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(restored, cmap='gray', vmin=0, vmax=255)
    axes[1, i].set_title(f'PSNR={psnr:.1f} dB', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('CLS Filter', fontsize=12)
axes[1, 0].set_ylabel('Restored', fontsize=12)

plt.suptitle('Effect of Regularization Parameter γ on CLS Restoration\n'
             'Small γ = noisy | Optimal γ = balanced | Large γ = over-smoothed',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Find optimal gamma and compare with Wiener

# CLS: sweep gamma
gamma_range = np.logspace(-5, 0, 200)
psnr_cls = [compute_psnr(original, cls_filter(G, H_turb, g, P_lap)) for g in gamma_range]
best_gamma = gamma_range[np.argmax(psnr_cls)]
best_psnr_cls = max(psnr_cls)

# Wiener: sweep K
K_range = np.logspace(-5, 0, 200)
psnr_wien = [compute_psnr(original, wiener_filter(G, H_turb, K)) for K in K_range]
best_K = K_range[np.argmax(psnr_wien)]
best_psnr_wien = max(psnr_wien)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogx(gamma_range, psnr_cls, 'b-', linewidth=2, label='CLS')
axes[0].semilogx(K_range, psnr_wien, 'r-', linewidth=2, label='Wiener')
axes[0].axvline(x=best_gamma, color='b', linestyle='--', alpha=0.5)
axes[0].axvline(x=best_K, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('Parameter (γ or K)', fontsize=12)
axes[0].set_ylabel('PSNR (dB)', fontsize=12)
axes[0].set_title('PSNR vs Regularization Parameter', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Compare best results
restored_cls = cls_filter(G, H_turb, best_gamma, P_lap)
restored_wien = wiener_filter(G, H_turb, best_K)

row = 128
axes[1].plot(original[row, :], 'k-', linewidth=1, alpha=0.5, label='Original')
axes[1].plot(restored_wien[row, :], 'r-', linewidth=1.5, label=f'Wiener ({best_psnr_wien:.1f} dB)')
axes[1].plot(restored_cls[row, :], 'b-', linewidth=1.5, label=f'CLS ({best_psnr_cls:.1f} dB)')
axes[1].set_title(f'Row {row} Profile Comparison', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlabel('Column')
axes[1].set_ylabel('Intensity')

plt.suptitle('CLS vs Wiener: Both Achieve Similar Optimal Performance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Best Wiener: K={best_K:.5f}, PSNR={best_psnr_wien:.1f} dB")
print(f"Best CLS:    γ={best_gamma:.5f}, PSNR={best_psnr_cls:.1f} dB")
print(f"Both methods achieve similar performance, but CLS requires less prior knowledge.")

---
## 3. Choosing γ: The Residual Constraint

The CLS filter has a built-in criterion for choosing $\gamma$:

The residual $r = g - H\hat{f}$ should satisfy:

$$\| r \|^2 = \| \eta \|^2 = M \cdot N \cdot \sigma_\eta^2$$

### Iterative procedure:
1. Start with an initial $\gamma$
2. Compute $\hat{f}$ using CLS
3. Compute $\| r \|^2 = \| g - H\hat{f} \|^2$
4. If $\| r \|^2 > \| \eta \|^2$: increase $\gamma$ (more smoothing)
5. If $\| r \|^2 < \| \eta \|^2$: decrease $\gamma$ (less smoothing)
6. Repeat until $\| r \|^2 \approx \| \eta \|^2$

This is a major advantage over Wiener: the **noise variance alone** determines the optimal $\gamma$.

In [ ]:
# Demonstrate the residual-based gamma selection

target_residual = M * N * noise_sigma**2  # expected noise power

gamma_range = np.logspace(-5, 0, 100)
residuals = []

for gamma in gamma_range:
    restored = cls_filter(G, H_turb, gamma, P_lap)
    F_hat = np.fft.fftshift(np.fft.fft2(restored))
    R = G - H_turb * F_hat
    residual = np.sum(np.abs(R)**2)
    residuals.append(residual)

residuals = np.array(residuals)

# Find gamma where residual matches target
idx_match = np.argmin(np.abs(residuals - target_residual))
gamma_auto = gamma_range[idx_match]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].semilogx(gamma_range, residuals, 'b-', linewidth=2)
axes[0].axhline(y=target_residual, color='r', linestyle='--', linewidth=2,
                label=f'Target: MNσ² = {target_residual:.0f}')
axes[0].axvline(x=gamma_auto, color='g', linestyle='--', alpha=0.7,
                label=f'γ = {gamma_auto:.5f}')
axes[0].set_xlabel('γ', fontsize=12)
axes[0].set_ylabel('||r||²', fontsize=12)
axes[0].set_title('Residual vs γ', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].imshow(degraded, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Degraded\nPSNR={compute_psnr(original, degraded):.1f} dB', fontsize=12)
axes[1].axis('off')

restored_auto = cls_filter(G, H_turb, gamma_auto, P_lap)
axes[2].imshow(restored_auto, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'CLS (auto γ={gamma_auto:.5f})\nPSNR={compute_psnr(original, restored_auto):.1f} dB', fontsize=12)
axes[2].axis('off')

plt.suptitle('Automatic γ Selection Using Residual Constraint\n'
             'Choose γ where ||g - H·f̂||² = M·N·σ²',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Automatic γ (from residual): {gamma_auto:.5f}, PSNR: {compute_psnr(original, restored_auto):.1f} dB")
print(f"Optimal γ (from oracle):     {best_gamma:.5f}, PSNR: {best_psnr_cls:.1f} dB")
print("The automatic selection is close to optimal — and requires NO original image!")

---
## 4. Geometric Mean Filter

A generalization that includes both inverse and Wiener filters as special cases:

$$\hat{F}(u,v) = \left[\frac{1}{H(u,v)}\right]^\alpha \left[\frac{H^*(u,v)}{|H(u,v)|^2 + \beta S_\eta/S_f}\right]^{1-\alpha} G(u,v)$$

Where:
- $\alpha = 1, \beta = 1$: **Inverse filter**
- $\alpha = 0, \beta = 1$: **Parametric Wiener filter**
- $\alpha = 0.5, \beta = 1$: **Geometric mean** of inverse and Wiener

The geometric mean filter allows fine-tuning between these extremes.

In [ ]:
# Implement and demonstrate the geometric mean filter

def geometric_mean_filter(G, H, alpha, K):
    """
    Geometric mean filter.
    W = (1/H)^alpha * (H*/(|H|^2 + K))^(1-alpha)
    
    alpha=1: inverse filter
    alpha=0: Wiener filter
    """
    H_safe = np.where(np.abs(H) > 1e-10, H, 1e-10 * np.exp(1j * np.angle(H)))
    
    # Inverse component
    inv_part = 1.0 / H_safe
    
    # Wiener component
    wien_part = np.conj(H) / (np.abs(H)**2 + K)
    
    # Use magnitude and phase separately for stability
    inv_mag = np.abs(inv_part)
    wien_mag = np.abs(wien_part)
    
    # Geometric mean of magnitudes
    W_mag = inv_mag**alpha * wien_mag**(1-alpha)
    W_phase = alpha * np.angle(inv_part) + (1-alpha) * np.angle(wien_part)
    W = W_mag * np.exp(1j * W_phase)
    
    # Clip to avoid extreme values
    W = np.where(np.abs(W) > 1e6, 0, W)
    
    F_hat = W * G
    restored = np.real(np.fft.ifft2(np.fft.ifftshift(F_hat)))
    return np.clip(restored, 0, 255)


alpha_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
K = 0.005  # fixed K

fig, axes = plt.subplots(1, 6, figsize=(22, 4))

for i, alpha in enumerate(alpha_values):
    restored = geometric_mean_filter(G, H_turb, alpha, K)
    psnr = compute_psnr(original, restored)
    
    axes[i].imshow(restored, cmap='gray', vmin=0, vmax=255)
    label = '(Wiener)' if alpha == 0 else '(Inverse)' if alpha == 1 else ''
    axes[i].set_title(f'α={alpha} {label}\nPSNR={psnr:.1f} dB', fontsize=10)
    axes[i].axis('off')

plt.suptitle('Geometric Mean Filter: Transition from Wiener (α=0) to Inverse (α=1)\n'
             'α controls the balance between noise suppression and deblurring',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("α=0 (Wiener): Best noise suppression, most regularized")
print("α=1 (Inverse): Most deblurring, most noise amplification")
print("Intermediate α: Tunable compromise")

---
## 5. Comprehensive Comparison: All Restoration Methods

Let's compare all the methods we've learned on the same degraded image.

In [ ]:
# Grand comparison on atmospheric turbulence + noise

# Create degraded image
k_turb = 0.005
H = atmospheric_turbulence_otf(M, N, k_turb)
F_orig = np.fft.fftshift(np.fft.fft2(original))

np.random.seed(42)
noise_sigma = 5
degraded = np.real(np.fft.ifft2(np.fft.ifftshift(H * F_orig)))
degraded += np.random.normal(0, noise_sigma, degraded.shape)
degraded = np.clip(degraded, 0, 255)
G = np.fft.fftshift(np.fft.fft2(degraded))

# Apply all methods
# 1. Inverse filter
restored_inv = inverse_filter(G, H, epsilon=0.01)

# 2. Truncated inverse
def truncated_inverse(G, H, D0):
    u = np.arange(M) - M/2
    v = np.arange(N) - N/2
    U, V = np.meshgrid(u, v, indexing='ij')
    D = np.sqrt(U**2 + V**2)
    F_hat = np.zeros_like(G, dtype=complex)
    mask = D <= D0
    H_safe = np.where(np.abs(H) > 1e-10, H, 1e-10)
    F_hat[mask] = G[mask] / H_safe[mask]
    return np.clip(np.real(np.fft.ifft2(np.fft.ifftshift(F_hat))), 0, 255)

# Find best D0 for truncated inverse
best_psnr_trunc = -np.inf
for D0_try in range(10, 128, 2):
    res = truncated_inverse(G, H, D0_try)
    p = compute_psnr(original, res)
    if p > best_psnr_trunc:
        best_psnr_trunc = p
        best_D0 = D0_try
restored_trunc = truncated_inverse(G, H, best_D0)

# 3. Wiener (optimal K)
K_search = np.logspace(-5, 0, 100)
psnrs_w = [compute_psnr(original, wiener_filter(G, H, K)) for K in K_search]
best_K = K_search[np.argmax(psnrs_w)]
restored_wien = wiener_filter(G, H, best_K)

# 4. CLS (optimal gamma)
gamma_search = np.logspace(-5, 0, 100)
psnrs_c = [compute_psnr(original, cls_filter(G, H, g, P_lap)) for g in gamma_search]
best_gamma = gamma_search[np.argmax(psnrs_c)]
restored_cls_opt = cls_filter(G, H, best_gamma, P_lap)

# Display all results
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

results = [
    (original, 'Original'),
    (degraded, f'Degraded\nPSNR={compute_psnr(original, degraded):.1f}'),
    (restored_inv, f'Pseudoinverse\nPSNR={compute_psnr(original, restored_inv):.1f}'),
    (restored_trunc, f'Truncated Inverse (D0={best_D0})\nPSNR={compute_psnr(original, restored_trunc):.1f}'),
    (restored_wien, f'Wiener (K={best_K:.4f})\nPSNR={compute_psnr(original, restored_wien):.1f}'),
    (restored_cls_opt, f'CLS (γ={best_gamma:.4f})\nPSNR={compute_psnr(original, restored_cls_opt):.1f}'),
]

for i, (img, title) in enumerate(results):
    row, col = i // 3, i % 3
    axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    axes[row, col].set_title(f'{title} dB', fontsize=11)
    axes[row, col].axis('off')

plt.suptitle('Comprehensive Comparison of All Restoration Methods\n'
             f'Degradation: Atmospheric Turbulence (k={k_turb}) + Gaussian Noise (σ={noise_sigma})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Quantitative comparison bar chart

methods = ['Degraded', 'Pseudo-\ninverse', 'Truncated\nInverse', 'Wiener', 'CLS']
psnrs = [
    compute_psnr(original, degraded),
    compute_psnr(original, restored_inv),
    compute_psnr(original, restored_trunc),
    compute_psnr(original, restored_wien),
    compute_psnr(original, restored_cls_opt),
]

colors = ['gray', 'red', 'orange', 'blue', 'green']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(methods, psnrs, color=colors, edgecolor='black', linewidth=0.5)

for bar, psnr in zip(bars, psnrs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{psnr:.1f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('PSNR (dB)', fontsize=13)
ax.set_title('Restoration Quality Comparison (PSNR)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, max(psnrs) + 5])
plt.tight_layout()
plt.show()

print("Ranking (best to worst):")
sorted_idx = np.argsort(psnrs)[::-1]
for rank, idx in enumerate(sorted_idx, 1):
    print(f"  {rank}. {methods[idx].replace(chr(10), ' ')}: {psnrs[idx]:.1f} dB")

In [ ]:
# Biomedical application: MRI-like phantom with motion degradation

def create_mri_phantom(size=256):
    """Create an MRI-like brain phantom."""
    img = np.zeros((size, size), dtype=np.float64)
    Y, X = np.mgrid[0:size, 0:size]
    cx, cy = size // 2, size // 2
    
    # Skull (outer bright ring)
    skull_outer = ((X-cx)/95)**2 + ((Y-cy)/85)**2 <= 1
    skull_inner = ((X-cx)/88)**2 + ((Y-cy)/78)**2 <= 1
    img[skull_outer & ~skull_inner] = 200
    
    # Brain tissue
    img[skull_inner] = 140
    
    # Gray matter regions
    gm1 = ((X-cx-20)/30)**2 + ((Y-cy-10)/40)**2 <= 1
    img[gm1 & skull_inner] = 170
    
    gm2 = ((X-cx+25)/25)**2 + ((Y-cy+15)/35)**2 <= 1
    img[gm2 & skull_inner] = 165
    
    # Ventricles (dark, CSF)
    vent1 = ((X-cx-8)/8)**2 + ((Y-cy)/20)**2 <= 1
    vent2 = ((X-cx+8)/8)**2 + ((Y-cy)/20)**2 <= 1
    img[vent1 | vent2] = 60
    
    # Small lesion
    lesion = (X-cx+35)**2 + (Y-cy-25)**2 <= 6**2
    img[lesion] = 220
    
    return img


phantom = create_mri_phantom(256)
M, N = phantom.shape

# Degrade with motion blur (simulating patient movement)
H_motion = motion_blur_otf(M, N, a=0.04, b=0.0)
F_phantom = np.fft.fftshift(np.fft.fft2(phantom))

np.random.seed(42)
degraded_mri = np.real(np.fft.ifft2(np.fft.ifftshift(H_motion * F_phantom)))
degraded_mri += np.random.normal(0, 3, degraded_mri.shape)
degraded_mri = np.clip(degraded_mri, 0, 255)

G_mri = np.fft.fftshift(np.fft.fft2(degraded_mri))

# Recompute P_lap for this image size
lap_padded_mri = np.zeros((M, N), dtype=np.float64)
for i in range(3):
    for j in range(3):
        ni = (i - 1) % M
        nj = (j - 1) % N
        lap_padded_mri[ni, nj] = laplacian[i, j]
P_lap_mri = np.fft.fftshift(np.fft.fft2(lap_padded_mri))

# Find optimal parameters
K_search = np.logspace(-5, 0, 100)
psnrs_w_mri = [compute_psnr(phantom, wiener_filter(G_mri, H_motion, K)) for K in K_search]
best_K_mri = K_search[np.argmax(psnrs_w_mri)]

gamma_search = np.logspace(-5, 0, 100)
psnrs_c_mri = [compute_psnr(phantom, cls_filter(G_mri, H_motion, g, P_lap_mri)) for g in gamma_search]
best_gamma_mri = gamma_search[np.argmax(psnrs_c_mri)]

restored_wien_mri = wiener_filter(G_mri, H_motion, best_K_mri)
restored_cls_mri = cls_filter(G_mri, H_motion, best_gamma_mri, P_lap_mri)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(phantom, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original MRI Phantom', fontsize=12)
axes[0].axis('off')

axes[1].imshow(degraded_mri, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'Motion Blur + Noise\nPSNR={compute_psnr(phantom, degraded_mri):.1f} dB', fontsize=12)
axes[1].axis('off')

axes[2].imshow(restored_wien_mri, cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'Wiener Filter\nPSNR={compute_psnr(phantom, restored_wien_mri):.1f} dB', fontsize=12)
axes[2].axis('off')

axes[3].imshow(restored_cls_mri, cmap='gray', vmin=0, vmax=255)
axes[3].set_title(f'CLS Filter\nPSNR={compute_psnr(phantom, restored_cls_mri):.1f} dB', fontsize=12)
axes[3].axis('off')

plt.suptitle('Biomedical Application: Restoring Motion-Degraded MRI Phantom',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Clinical impact: Motion blur during MRI can obscure lesions and tissue boundaries.")
print("Wiener and CLS filters can recover diagnostic information from degraded scans.")

---
## 6. Summary of All Restoration Methods (Chapter 5)

### Noise-Only Restoration ($H = \delta$, so $g = f + \eta$):

| Method | Best For | Key Property |
|---|---|---|
| Arithmetic mean | Gaussian noise | Simple, blurs edges |
| Geometric mean | Gaussian noise | Better detail preservation |
| Harmonic mean | Salt noise | Fails for pepper |
| Contraharmonic ($Q$) | Salt OR pepper | Sign of $Q$ critical |
| **Median** | **Salt-and-pepper** | **Gold standard for impulse noise** |
| Alpha-trimmed mean | Mixed noise | Compromise of mean and median |
| **Adaptive median** | **Heavy S&P** | **Variable window, preserves detail** |
| Adaptive local | Gaussian noise | Variable smoothing strength |
| Notch filter | Periodic noise | Frequency domain, surgical |

### Blur + Noise Restoration ($g = h \star f + \eta$):

| Method | Formula | Pros | Cons |
|---|---|---|---|
| Inverse | $G/H$ | Simple | Noise explosion at $H=0$ |
| Truncated inverse | $G/H$ within $D_0$ | Better than naive | $D_0$ hard to choose |
| Pseudoinverse | $G/H$ where $|H|>\epsilon$ | Avoids zeros | $\epsilon$ hard to choose |
| **Wiener** | $\frac{H^*}{|H|^2+K} G$ | **Optimal MSE** | Needs $S_\eta/S_f$ |
| **CLS** | $\frac{H^*}{|H|^2+\gamma|P|^2} G$ | **Auto-tunable** | Needs $\sigma_\eta^2$ only |
| Geometric mean | $\alpha$-blend of inverse/Wiener | Flexible | Two parameters |

---
## Summary

What we learned:

1. **Constrained Least Squares (CLS)** filter minimizes $\|\nabla^2 \hat{f}\|^2$ subject to $\|g - H\hat{f}\|^2 = \|\eta\|^2$. The solution is:
   $\hat{F} = \frac{H^*}{|H|^2 + \gamma|P|^2} G$

2. **CLS advantage over Wiener**: requires only **noise variance** $\sigma_\eta^2$ (not the signal power spectrum). The parameter $\gamma$ can be found automatically from the residual constraint.

3. **The Laplacian** $P(u,v)$ acts as a frequency-dependent regularizer that penalizes high frequencies more than low, naturally suppressing noise.

4. **Geometric mean filter** provides a tunable transition between inverse ($\alpha=1$) and Wiener ($\alpha=0$) filters.

5. In the comprehensive comparison, **Wiener and CLS achieve the best results**, significantly outperforming inverse and truncated inverse filters.

6. **No single restoration method works for all situations**. The choice depends on:
   - Type of noise (Gaussian, impulse, periodic)
   - Presence of blur (with or without degradation function)
   - Available prior knowledge (noise variance, PSF, power spectra)
   - Computational requirements

7. In **biomedical imaging**, Wiener and CLS filters are the most widely used restoration methods for recovering diagnostic information from degraded medical images.